# Try ReFactX v3

Tests the new `PatternConstrainedGeneration` architecture with:
- **Fact:** triple retrieval with sentinel (`no further records>`)
- **count_branches:** tool for counting KB entries
- **`</think>`** thinking blocks with cache reset

Uses Qwen/Qwen3.5-4B and the v3 prompt.

In [1]:
import sys
sys.settrace(None)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
from pathlib import Path
import json
import torch
_cuda_lib = Path(torch.__file__).resolve().parent.parent / 'nvidia' / 'cu13' / 'lib'
if _cuda_lib.is_dir():
    os.environ['LD_LIBRARY_PATH'] = f"{_cuda_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
import ctypes
if _cuda_lib.is_dir():
    ctypes.CDLL(str(_cuda_lib / 'libnvJitLink.so.13'), mode=ctypes.RTLD_GLOBAL)
import time
import ipywidgets as widgets
from dotenv import load_dotenv
load_dotenv()

from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoProcessor, AutoModelForCausalLM, TextStreamer
from transformers import ProcessorMixin
from peft import PeftModel

import refactx
from refactx.generate import (
    patch_model, CONSTRAINED_STATES,
    get_constrained_logits_processor,
)

/opt/conda/envs/trl/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [4]:
MODEL = 'Qwen/Qwen3.5-4B'
ADAPTER = '/workspace/data/sft_output_qwen35_4b'
INDEX = os.environ.get('POSTGRES_URL', '../indexes/simple_index.txt.gz')
#PROMPT_PATH = '../prompts/prompt_qwen36_mini2_v3.json'
PROMPT_DIR = Path('../prompts')

In [5]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

'cuda'

## Load Model, Index, and Prompt

In [6]:
processor = AutoProcessor.from_pretrained(MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map='auto',
    dtype=torch.float16,
)
if ADAPTER:
    model = PeftModel.from_pretrained(base_model, ADAPTER, is_trainable=False)
    model = model.merge_and_unload(safe_merge=True)
model.eval()

tokenizer = processor
streamer = TextStreamer(tokenizer.tokenizer)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [7]:
index = refactx.load_index(INDEX, tokenizer=tokenizer)
print(f'Index type: {type(index).__name__}')

Applying index config...
Index type: PostgresTrieIndex


In [8]:
prompt_paths = sorted(str(path) for path in PROMPT_DIR.iterdir()
                      if path.suffix.lower() in {'.yaml', '.yml', '.json', '.txt'})
prompt_selector = widgets.Dropdown(
    options=prompt_paths,
    value=next((path for path in prompt_paths
                if path.endswith('prompt_qwen36_angular3.yaml')), prompt_paths[0]),
    description='Prompt:',
    layout=widgets.Layout(width='700px'))
display(prompt_selector)
def load_selected_prompt(change=None):
    global prompt_messages, prompt_path
    prompt_path = prompt_selector.value
    prompt_messages = refactx.load_prompt(prompt_path)
    print(f'Prompt loaded from {prompt_path}')



Dropdown(description='Prompt:', layout=Layout(width='700px'), options=('../prompts/prompt_qwen36_angular2.yaml…

In [9]:
prompt_selector.observe(load_selected_prompt, names='value')
load_selected_prompt()

Prompt loaded from ../prompts/prompt_qwen36_angular2.yaml


In [11]:
patch_model(model)

In [12]:
logits_processor = get_constrained_logits_processor(
    tokenizer, index, num_beams=1, num_batches=1,
    sentinel=True)

## Helper Functions

In [13]:
def _tokenize(tok, text):
    if isinstance(tok, ProcessorMixin):
        return tok.tokenizer(text, return_tensors='pt')
    return tok(text, return_tensors='pt')

def _decode(tok, ids):
    if isinstance(tok, ProcessorMixin):
        return tok.tokenizer.decode(ids, skip_special_tokens=True)
    return tok.decode(ids, skip_special_tokens=True)

def ask(question, max_new_tokens=800, thinking=False):
    """Ask a question using the currently selected prompt."""

    logits_processor[0].reset_states()
    
    full_prompt = refactx.apply_prompt_template(
        tokenizer, prompt_template=prompt_messages, question=question, enable_thinking=thinking)
    inputs = _tokenize(tokenizer, full_prompt).to(model.device)

    # Fresh constrained processor each call. The shared factory registers the
    # fact and count patterns and forces a multi-token EOT.

    model.eval()
    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            logits_processor=logits_processor,
            max_new_tokens=max_new_tokens,
            streamer=streamer,
            do_sample=True,
            top_p=0.8,
            min_p=0.0,
            temperature=0.7,
            #presence_penalty=1.5,
            repetition_penalty=1.0,
            num_beams=1,
            num_return_sequences=1,
            use_cache=True,
            eos_token_id=tokenizer.tokenizer.eos_token_id
        )
    elapsed = time.time() - start

    state = CONSTRAINED_STATES.states[0][0]
    text = _decode(tokenizer, out[0][inputs.input_ids.shape[1]:])
    facts = state.generated_triples
    facts_str = [_decode(tokenizer, t) for t in facts]

    print(f'\n--- Facts ({len(facts_str)}) ---')
    for i, f in enumerate(facts_str):
        print(f'  {i}: {f}')

    print(f'--- History ({len(state.generation_history)}) ---')
    for i, g in enumerate(state.generation_history):
        cls_name = type(g).__name__
        if hasattr(g, 'completed_with_sentinel'):
            print(f'  [{i}] {cls_name} sentinel={g.completed_with_sentinel}')
        elif hasattr(g, 'called'):
            print(f'  [{i}] {cls_name} called={g.called}')
        else:
            print(f'  [{i}] {cls_name}')

    print(f'Elapsed: {elapsed:.2f}s')
    torch.cuda.empty_cache()
    return text, facts_str

## Test 1: Multi-hop Reasoning

Ask a question that requires two `Fact:` lookups.

In [14]:
torch.cuda.empty_cache()

In [15]:
# The selected prompt is loaded in the load-prompt cell above.

In [16]:
enable_thinking = False

In [17]:
text, facts = ask('When was the director of Slumdog Millionaire born?', thinking=enable_thinking)

<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

In [ ]:
# tool basato su llm / policy / prediction

In [23]:
text, facts = ask('who is older? brad pitt or johnny depp?', thinking=enable_thinking)

<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

In [ ]:
text, facts = ask('Which is the river that flows in the city where there is the eiffel tower?', thinking=enable_thinking)

## Test 2: Counting

Ask a "how many" question to trigger `count_branches:`.

In [ ]:
text, facts = ask('How many countries share a border with France?', thinking=True)

## Test 3: Exhausted Records (Sentinel)

Ask a question that requires enumerating all objects until `no further records>`.

In [14]:
text, facts = ask(
    'Which countries share a border with Brazil?',
    max_new_tokens=1200)

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

## Test 4: Thinking + Facts

Ask a comparison question that triggers `<think>` reasoning followed by facts.

In [15]:
text, facts = ask(
    'Is Johnny Depp older than Brad Pitt?',
    max_new_tokens=1200)

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation pair have been retrieved.

## Counting
When you need to count how many objects exist for a subj

## Interactive

Edit the question below to test interactively.

In [15]:
MY_QUESTION = 'Who was the first person to walk on the moon?'

text, facts = ask(MY_QUESTION, max_new_tokens=1200)

Setting `pad_token_id` to `eos_token_id`:248046 for open-end generation.


<|im_start|>system
# Fact-Grounded Question Answering

## Role
You are a question-answering system that answers using only verified facts retrieved from a knowledge base.

## Core Principle
All answers must be grounded in retrieved facts. Do not use prior or parametric knowledge when producing answers.

## Reasoning
Reason about how to answer the question. You may use `<think>` blocks for step-by-step reasoning.

Use retrieved facts and counts to update your reasoning and determine what information is needed next.

## Fact Retrieval
When you need information from the knowledge base, call `<fact>` inside a `<think>` block. The retrieval command must be part of the thinking process, not emitted as a standalone block.

The system will return verified knowledge-base facts. You may use multiple `<fact>` calls inside `<think>` blocks when needed.

Only use retrieved facts as evidence for the final answer.

When the system returns `<no further records>`, all objects for that subject-relation 